In [1]:
# Deep Learning with Text Features - Notebook 04
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dropout, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

print("🧠 Starting Deep Learning with Text Features...")
print("=" * 60)

# Load the data
print("📊 Loading data...")
train_data = pd.read_csv('../data/processed/train_data.csv')
val_data = pd.read_csv('../data/processed/val_data.csv')
test_data = pd.read_csv('../data/processed/test_data.csv')

print(f"Train: {train_data.shape}, Validation: {val_data.shape}, Test: {test_data.shape}")

# Prepare text data
print("\n🔤 Preparing text data...")
text_columns = ['description', 'requirements', 'company_profile']

# Combine text features
train_data['combined_text'] = train_data[text_columns].fillna('').apply(lambda x: ' '.join(x), axis=1)
val_data['combined_text'] = val_data[text_columns].fillna('').apply(lambda x: ' '.join(x), axis=1)
test_data['combined_text'] = test_data[text_columns].fillna('').apply(lambda x: ' '.join(x), axis=1)

print(f"Sample text length: {len(train_data['combined_text'].iloc[0])} characters")

# Tokenize text
print("📝 Tokenizing text...")
tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(train_data['combined_text'])

# Convert text to sequences
X_train_text = tokenizer.texts_to_sequences(train_data['combined_text'])
X_val_text = tokenizer.texts_to_sequences(val_data['combined_text'])
X_test_text = tokenizer.texts_to_sequences(test_data['combined_text'])

# Pad sequences
max_length = 200  # Limit sequence length
X_train_text = pad_sequences(X_train_text, maxlen=max_length, padding='post', truncating='post')
X_val_text = pad_sequences(X_val_text, maxlen=max_length, padding='post', truncating='post')
X_test_text = pad_sequences(X_test_text, maxlen=max_length, padding='post', truncating='post')

print(f"Text sequences shape: {X_train_text.shape}")

# Prepare numeric features
print("\n🔢 Preparing numeric features...")
numeric_features = ['telecommuting', 'has_company_logo', 'has_questions']

X_train_numeric = train_data[numeric_features].values
X_val_numeric = val_data[numeric_features].values
X_test_numeric = test_data[numeric_features].values

# Prepare target
y_train = train_data['fraudulent'].values
y_val = val_data['fraudulent'].values
y_test = test_data['fraudulent'].values

print(f"Numeric features shape: {X_train_numeric.shape}")
print(f"Target shape: {y_train.shape}")

# Build LSTM model
print("\n🏗️ Building LSTM model...")

# Text input branch
text_input = Input(shape=(max_length,), name='text_input')
embedding = Embedding(input_dim=10000, output_dim=128, name='embedding')(text_input)
lstm = LSTM(64, name='lstm')(embedding)
text_dropout = Dropout(0.3, name='text_dropout')(lstm)

# Numeric input branch
numeric_input = Input(shape=(len(numeric_features),), name='numeric_input')
numeric_dense = Dense(16, activation='relu', name='numeric_dense')(numeric_input)
numeric_dropout = Dropout(0.3, name='numeric_dropout')(numeric_dense)

# Combine branches
combined = Concatenate(name='concat')([text_dropout, numeric_dropout])
dense1 = Dense(32, activation='relu', name='dense1')(combined)
final_dropout = Dropout(0.3, name='final_dropout')(dense1)
output = Dense(1, activation='sigmoid', name='output')(final_dropout)

# Create model
model = Model(inputs=[text_input, numeric_input], outputs=output)
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

print("✅ Model built successfully!")
model.summary()

# Train model
print("\n🎯 Training model...")
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    [X_train_text, X_train_numeric], y_train,
    validation_data=([X_val_text, X_val_numeric], y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate model
print("\n📊 Evaluating model...")
test_loss, test_accuracy, test_precision, test_recall = model.evaluate(
    [X_test_text, X_test_numeric], y_test, verbose=0
)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")

# Make predictions
y_pred_proba = model.predict([X_test_text, X_test_numeric])
y_pred = (y_pred_proba > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Save model
model.save('../models/lstm_text_model.h5')
print("💾 Model saved to: ../models/lstm_text_model.h5")

print("\n" + "=" * 60)
print("✅ Deep Learning with text features complete!")
print("🚀 Next: Compare with basic ML models and choose the best!")

🧠 Starting Deep Learning with Text Features...
📊 Loading data...
Train: (12516, 18), Validation: (2682, 18), Test: (2682, 18)

🔤 Preparing text data...
Sample text length: 5694 characters
📝 Tokenizing text...
Text sequences shape: (12516, 200)

🔢 Preparing numeric features...
Numeric features shape: (12516, 3)
Target shape: (12516,)

🏗️ Building LSTM model...
✅ Model built successfully!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ text_input (InputLayer)       │ (None, 200)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 200, 128)          │       1,280,000 │ text_input[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ numeric_input (InputLayer)    │ (None, 3)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ (None, 64)                │          49,408 │ embedding[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ numeric_dense (Dense)         │ (None, 16)                │              64 │ numeric_input[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ text_dropout (Dropout)        │ (None, 64)                │               0 │ lstm[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ numeric_dropout (Dropout)     │ (None, 16)                │               0 │ numeric_dense[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concat (Concatenate)          │ (None, 80)                │               0 │ text_dropout[0][0],        │
│                               │                           │                 │ numeric_dropout[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense1 (Dense)                │ (None, 32)                │           2,592 │ concat[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ final_dropout (Dropout)       │ (None, 32)                │               0 │ dense1[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ output (Dense)                │ (None, 1)                 │              33 │ final_dropout[0][0]        │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,332,097 (5.08 MB)

 Trainable params: 1,332,097 (5.08 MB)

 Non-trainable params: 0 (0.00 B)


🎯 Training model...
Epoch 1/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 54s 126ms/step - accuracy: 0.9484 - loss: 0.2052 - precision: 0.0455 - recall: 0.0033 - val_accuracy: 0.9515 - val_loss: 0.1599 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 46s 117ms/step - accuracy: 0.9541 - loss: 0.1594 - precision: 0.9444 - recall: 0.0561 - val_accuracy: 0.9567 - val_loss: 0.1475 - val_precision: 0.7917 - val_recall: 0.1462
Epoch 3/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 46s 118ms/step - accuracy: 0.9620 - loss: 0.1136 - precision: 0.8421 - recall: 0.2640 - val_accuracy: 0.9594 - val_loss: 0.1429 - val_precision: 0.8889 - val_recall: 0.1846
Epoch 4/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 48s 121ms/step - accuracy: 0.9697 - loss: 0.0840 - precision: 0.9416 - recall: 0.3993 - val_accuracy: 0.9545 - val_loss: 0.1473 - val_precision: 0.5909 - val_recall: 0.2000
Epoch 5/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 47s 120ms/step - accuracy: 0.9736 - loss: 0.0623 - precision: 0.8809 - recall: 


Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      2552
           1       0.83      0.23      0.36       130

    accuracy                           0.96      2682
   macro avg       0.90      0.61      0.67      2682
weighted avg       0.96      0.96      0.95      2682

Confusion Matrix:
[[2546    6]
 [ 100   30]]
💾 Model saved to: ../models/lstm_text_model.h5

✅ Deep Learning with text features complete!
🚀 Next: Compare with basic ML models and choose the best!
